# 🔥 Day 01a: PyTorch Tensors & Autograd — The Foundation

---

## 🎯 What You'll Master Today
- Tensor creation, operations, device management
- Autograd — automatic differentiation
- NumPy ↔ Tensor conversion
- GPU acceleration basics

**Goal:** Work with tensors and autograd fluently from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  PyTorch = NumPy + Autograd + GPU                                ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  NumPy array          →  torch.Tensor                           ║
║  np.zeros()           →  torch.zeros()                          ║
║  arr.reshape()        →  tensor.view() / tensor.reshape()       ║
║  np.dot(a, b)         →  torch.matmul(a, b) / a @ b            ║
║                                                                   ║
║  AUTOGRAD:                                                       ║
║    x = tensor(requires_grad=True)                               ║
║    y = f(x)                                                      ║
║    y.backward()  → x.grad has dy/dx                             ║
║                                                                   ║
║  GPU:                                                            ║
║    tensor.to('cuda') / tensor.cuda()                            ║
║    model.to('cuda')                                              ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import torch
import numpy as np

# ============================================================
# 1. Tensor Creation — All the ways
# ============================================================

# From data
t1 = torch.tensor([1, 2, 3])                # int64 by default
t2 = torch.tensor([1.0, 2.0, 3.0])          # float32 by default
t3 = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

# Standard constructors
z = torch.zeros(3, 4)          # 3x4 of zeros
o = torch.ones(2, 3)           # 2x3 of ones
r = torch.randn(3, 3)          # normal distribution
u = torch.rand(3, 3)           # uniform [0, 1)
e = torch.eye(3)               # identity matrix
a = torch.arange(0, 10, 2)     # [0, 2, 4, 6, 8]
l = torch.linspace(0, 1, 5)    # [0, 0.25, 0.5, 0.75, 1.0]

# Like existing tensor (same shape, dtype, device)
z_like = torch.zeros_like(t3)
r_like = torch.randn_like(t3)

print(f"tensor:   {t1}  dtype={t1.dtype}")
print(f"float:    {t2}  dtype={t2.dtype}")
print(f"randn:    shape={r.shape}")
print(f"arange:   {a}")
print(f"linspace: {l}")

In [ ]:
# ============================================================
# 2. Tensor Operations — NumPy-like
# ============================================================

a = torch.tensor([[1.0, 2], [3, 4]])
b = torch.tensor([[5.0, 6], [7, 8]])

# Element-wise
print(f"a + b:\n{a + b}")
print(f"a * b:\n{a * b}")  # element-wise, NOT matrix multiply

# Matrix multiply
print(f"a @ b:\n{a @ b}")                    # preferred
print(f"matmul:\n{torch.matmul(a, b)}")       # same

# Reshaping
t = torch.arange(12)
print(f"\noriginal: {t.shape}")
print(f"view:     {t.view(3, 4).shape}")       # contiguous only
print(f"reshape:  {t.reshape(3, 4).shape}")    # works always
print(f"unsqueeze: {t.unsqueeze(0).shape}")    # add dim: (12,) → (1, 12)
print(f"squeeze:   {t.unsqueeze(0).squeeze().shape}")  # remove dim

# Aggregations
x = torch.tensor([[1.0, 2, 3], [4, 5, 6]])
print(f"\nsum: {x.sum()}, mean: {x.mean()}, max: {x.max()}")
print(f"sum(dim=0): {x.sum(dim=0)}")   # along batch
print(f"sum(dim=1): {x.sum(dim=1)}")   # along features
print(f"argmax(dim=1): {x.argmax(dim=1)}")

In [ ]:
# ============================================================
# 3. NumPy ↔ Tensor — Zero-copy when possible
# ============================================================

# NumPy → Tensor (shares memory on CPU!)
np_arr = np.array([1.0, 2.0, 3.0])
tensor = torch.from_numpy(np_arr)    # shares memory
tensor2 = torch.tensor(np_arr)       # copies data

np_arr[0] = 99
print(f"from_numpy (shared): {tensor}")  # changed!
print(f"tensor() (copied):   {tensor2}")  # NOT changed

# Tensor → NumPy
t = torch.tensor([4.0, 5.0, 6.0])
n = t.numpy()              # shares memory (CPU only)
n2 = t.detach().numpy()    # safe: detach from graph first

print(f"\nTensor → numpy: {n}")

print("\n💡 from_numpy: shared memory. torch.tensor(): copy.")
print("   For GPU tensors: tensor.cpu().numpy()")

In [ ]:
# ============================================================
# 4. Autograd — Automatic Differentiation
# ============================================================

# requires_grad=True tells PyTorch to track operations
x = torch.tensor(3.0, requires_grad=True)

# Forward: y = x² + 2x + 1
y = x**2 + 2*x + 1

# Backward: compute dy/dx
y.backward()

# dy/dx = 2x + 2 = 2(3) + 2 = 8
print(f"x = {x.item()}")
print(f"y = x² + 2x + 1 = {y.item()}")
print(f"dy/dx = {x.grad.item()} (expected: 8)")

# ---- Multi-variable ----
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

z = a**2 * b + b**3
z.backward()

print(f"\nz = a²b + b³")
print(f"dz/da = 2ab = {a.grad.item()} (expected: {2*2*3})")
print(f"dz/db = a² + 3b² = {b.grad.item()} (expected: {4 + 27})")

In [ ]:
# ============================================================
# 5. Autograd in Practice — Gradient Descent
# ============================================================

# Learn y = 3x + 5 from data
torch.manual_seed(42)
X = torch.linspace(-5, 5, 100)
y_true = 3 * X + 5 + torch.randn(100) * 0.5  # with noise

# Parameters to learn
w = torch.tensor(0.0, requires_grad=True)
b_param = torch.tensor(0.0, requires_grad=True)
lr = 0.01

for epoch in range(100):
    # Forward
    y_pred = w * X + b_param
    loss = ((y_pred - y_true)**2).mean()
    
    # Backward
    loss.backward()
    
    # Update (no_grad: don't track this operation)
    with torch.no_grad():
        w -= lr * w.grad
        b_param -= lr * b_param.grad
    
    # Zero gradients (IMPORTANT: gradients accumulate by default!)
    w.grad.zero_()
    b_param.grad.zero_()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | w={w.item():.3f}, b={b_param.item():.3f}")

print(f"\nLearned: y = {w.item():.3f}x + {b_param.item():.3f}")
print(f"True:    y = 3.000x + 5.000")

print("\n💡 Key pattern: forward → backward → update (no_grad) → zero_grad")

In [ ]:
# ============================================================
# 6. Device Management & Common Gotchas
# ============================================================

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Move tensors
t = torch.randn(3, 3)
t_gpu = t.to(device)       # to GPU (or stays on CPU)
# t_cpu = t_gpu.cpu()      # back to CPU

# Common gotchas
print("\n⚠️ Common PyTorch Gotchas:")
print("1. Gradients accumulate → always zero_grad()")
print("2. in-place ops on requires_grad tensors → use .data or no_grad()")
print("3. GPU tensor can't .numpy() → .cpu().numpy()")
print("4. view() needs contiguous memory → use .contiguous().view()")
print("5. .item() to get Python scalar from 0-d tensor")

# Detach from graph
x = torch.tensor(3.0, requires_grad=True)
y = x * 2
y_detached = y.detach()  # no gradient tracking
print(f"\ny.requires_grad: {y.requires_grad}")
print(f"y_detached.requires_grad: {y_detached.requires_grad}")

---

## 🗺️ PyTorch Tensors Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  PyTorch TENSOR ESSENTIALS                                      ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  CREATE:                                                         ║
║    torch.tensor()  .zeros()  .ones()  .randn()  .arange()      ║
║    torch.from_numpy()  torch.*_like()                           ║
║                                                                   ║
║  SHAPE:                                                          ║
║    .view()  .reshape()  .unsqueeze()  .squeeze()  .permute()   ║
║    .transpose()  .contiguous()  .flatten()                      ║
║                                                                   ║
║  AUTOGRAD:                                                       ║
║    requires_grad=True → .backward() → .grad                    ║
║    torch.no_grad() for updates                                   ║
║    .zero_() to clear gradients                                   ║
║    .detach() to stop tracking                                    ║
║                                                                   ║
║  DEVICE:                                                         ║
║    .to(device)  .cuda()  .cpu()                                 ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is autograd? | PyTorch automatically computes gradients by building a computation graph. Call .backward() to get .grad |
| 2 | view() vs reshape()? | view() requires contiguous memory (fast, no copy). reshape() always works (may copy) |
| 3 | Why zero_grad()? | Gradients accumulate by default in PyTorch. Must zero before each backward pass |
| 4 | from_numpy vs tensor()? | from_numpy shares memory (changes affect both). tensor() copies data (independent) |
| 5 | When to use torch.no_grad()? | During inference and weight updates. Saves memory by not tracking computation graph |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Tensor = NumPy array + autograd + GPU support        │
## │  • requires_grad=True → backward() → .grad             │
## │  • Training loop: forward → loss → backward → update   │
## │  • Always zero_grad() before backward()                 │
## │  • .to(device) moves everything to GPU                  │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **nn.Module** — build real neural networks